In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os


In [ ]:
OUT_DIR = "/content/drive/MyDrive/mango_models"   # where models were saved
IMG_SIZE = 224


In [ ]:
# Load level-1 part classifier
part_model = keras.models.load_model(os.path.join(OUT_DIR, "part_model.keras"))

# Load level-2 disease models
blossom_model = keras.models.load_model(os.path.join(OUT_DIR, "blossom_model.keras"))
fruit_model   = keras.models.load_model(os.path.join(OUT_DIR, "fruit_model.keras"))
leaf_model    = keras.models.load_model(os.path.join(OUT_DIR, "leaf_model.keras"))
stem_model    = keras.models.load_model(os.path.join(OUT_DIR, "stem_model.keras"))


In [ ]:
import json

with open(os.path.join(OUT_DIR, "hierarchy_meta.json"), "r") as f:
    meta = json.load(f)

part_class_names = meta["part_classes"]
blossom_classes = meta["blossom_classes"]
fruit_classes   = meta["fruit_classes"]
leaf_classes    = meta["leaf_classes"]
stem_classes    = meta["stem_classes"]


In [ ]:
part_to_model = {
    "Blossom": (blossom_model, blossom_classes),
    "Fruit":   (fruit_model, fruit_classes),
    "Leaf":    (leaf_model, leaf_classes),
    "Stem":    (stem_model, stem_classes),
}


In [ ]:
def hierarchical_predict(image_path):
    img = keras.preprocessing.image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras.preprocessing.image.img_to_array(img)
    arr = np.expand_dims(arr, 0)
    arr_pre = tf.keras.applications.efficientnet_v2.preprocess_input(arr)

    # Level 1: Part Prediction
    part_preds = part_model.predict(arr_pre)
    part_idx = int(np.argmax(part_preds, axis=1)[0])
    part_name = part_class_names[part_idx]
    part_conf = float(np.max(part_preds))

    # Level 2: Disease Prediction
    model2, class_list = part_to_model[part_name]
    disease_preds = model2.predict(arr_pre)
    disease_idx = int(np.argmax(disease_preds, axis=1)[0])
    disease_name = class_list[disease_idx]
    disease_conf = float(np.max(disease_preds))

    return {
        "part": part_name,
        "part_confidence": part_conf,
        "disease": disease_name,
        "disease_confidence": disease_conf
    }


In [ ]:
result = hierarchical_predict("/content/yellow-mango.jpeg")
print(result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 604ms/step
{'part': 'Fruit', 'part_confidence': 0.8009034395217896, 'disease': 'Fruit- Black mold rot', 'disease_confidence': 0.4052414000034332}


In [ ]:
result = hierarchical_predict("/content/Healthy (22).jpg")
print(result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 976ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
{'part': 'Fruit', 'part_confidence': 0.7747657299041748, 'disease': 'Fruit-healthy', 'disease_confidence': 0.762543797492981}
